In [1]:
import pandas as pd
import numpy as np
import sys
parent_path = '/mnt/digphat/syntheticDataBenchmark/Chuong_pipeline'
sys.path.append(parent_path)
from SynOmics.metrics.fidelity.UnivariateSimilarity import UnivariateSimilarity
from SynOmics.processing.metadata import MetaData
import os
from SynOmics.processing.postprocessing import post_masking


os.makedirs("TestUniGC", exist_ok = True)
or_data = pd.read_csv("original_data.csv", index_col = 0)

def move_last_column_to_first(df):
    # 1. Get the list of all column names
    cols = df.columns.tolist()

    # 2. Identify the last column (which we want to move)
    last_col = cols[-1]

    # 3. Create a new order for the columns: last column first, followed by all other original columns
    new_col_order = [last_col] + cols[:-1]

    # 4. Reindex the DataFrame using the new column order
    df = df[new_col_order]
    
    return df

or_data_0 = move_last_column_to_first(or_data)

In [4]:
or_data["BR"].value_counts()

BR
PD    56
PR    31
CR    16
SD    16
MR     2
Name: count, dtype: int64

In [5]:
or_data["PD vs Responder"].value_counts()

PD vs Responder
Progressor    56
Responder     47
SD/MR         18
Name: count, dtype: int64

In [13]:
masked_or_data.iloc[:,0:48]

,PD vs Responder,total_muts,nonsyn_muts,clonal_muts,subclonal_muts,heterogeneity,total_neoantigens,CNA_prop,gender,monthsBiopsyPreTx,...,biopsyContext,daysBiopsyToPD1,daysBiopsyAfterIpiStart,purity,ploidy,UV,Alkylating,Cosmic,MHCI_Amp,TAP2_Amp
Patient,,,,,,,,,,,,,,,,,,,,,
Patient1,Progressor,34.0,22.0,12.0,10.0,0.454545,49.0,0.321417,0.0,2.8,...,3.0,-84.0,noIpi,0.92,1.73,0.509426,2.514556e+00,28.973872,0,0
Patient10,Responder,96.0,71.0,48.0,22.0,0.314286,230.0,0.391384,0.0,0.4,...,3.0,-12.0,postIpi,0.83,1.84,35.883346,6.810735e+00,42.303101,0,0
Patient100,Responder,200.0,126.0,98.0,24.0,0.196721,301.0,0.029447,0.0,3.1,...,3.0,-94.0,postIpi,0.11,2.17,134.899883,1.269846e+01,33.400423,0,0
Patient102,SD/MR,370.0,246.0,215.0,26.0,0.107884,825.0,0.169389,1.0,2.1,...,3.0,-64.0,noIpi,0.70,3.24,281.460930,2.405394e+01,42.484586,0,0
Patient105,SD/MR,185.0,125.0,85.0,23.0,0.212963,334.0,0.394306,0.0,0.7,...,3.0,-22.0,noIpi,0.86,2.42,95.469481,1.752214e+01,53.005224,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Patient9,Progressor,911.0,652.0,562.0,80.0,0.124611,1996.0,0.111034,0.0,1.3,...,3.0,-39.0,noIpi,0.88,2.02,637.426283,3.637034e+01,146.198830,0,0
Patient94,Progressor,414.0,268.0,227.0,21.0,0.084677,929.0,0.902095,0.0,1.0,...,3.0,-31.0,postIpi,0.85,1.13,339.824541,2.280000e-08,50.175395,0,0
Patient96,Responder,959.0,649.0,588.0,50.0,0.078370,2136.0,0.196499,1.0,2.7,...,3.0,-81.0,noIpi,0.82,2.22,851.443224,2.373497e+00,61.187771,0,0


In [30]:
## AllDummy
masked_or_data = post_masking(or_data_0)
masked_or_data = masked_or_data.rename(columns={"PD vs Responder": "PD_vs_Responder"})
syn_1 = pd.read_csv("gaussiancopula42_AllDummy/gaussiancopula_42_AllDummy.csv", index_col = 0)
syn_1 = move_last_column_to_first(syn_1)
masked_syn_1 = post_masking(syn_1)
masked_syn_1 = masked_syn_1.rename(columns={"PD vs Responder": "PD_vs_Responder"})
metadata_1 = MetaData.get_metadata(data = masked_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = None)
uni_comparison_1 = UnivariateSimilarity(output_dir = "TestUniGC/AllDummy", logger_name = "AllDummy")
scores = uni_comparison_1.get_univariate_score(
original_data = masked_or_data.iloc[:,0:48], 
synthetic_data=masked_syn_1.iloc[:,0:48], 
metadata=metadata_1)
scores_df_1 = uni_comparison_1.get_detail_df()
scores_df_1.to_csv("TestUniGC/AllDummy/alldummy.csv", index = False)

2025-11-26 21:39:39 - DEBUG - Standalone logger initialized successfully.
2025-11-26 21:39:39 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 1677.18it/s]

2025-11-26 21:39:39 - INFO - Univariate similarity score: 0.9089334947969272
2025-11-26 21:39:39 - INFO - Details DataFrame saved to TestUniGC/AllDummy/Detail_score_AllDummy.csv


2025-11-26 21:39:41 - INFO - Histogram figure saved to TestUniGC/AllDummy/AllDummy.png


In [38]:
#BR ordinal no new columns

masked_or_data = post_masking(or_data_0)
masked_or_data_2 = masked_or_data.drop(columns=["PD vs Responder"])
syn_2 = pd.read_csv("gaussiancopula_42.csv", index_col = 0)
masked_syn_2 = post_masking(syn_2)

ordinal_features = ["BR", "Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext"]
metadata_2 = MetaData.get_metadata(data = masked_or_data_2, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ordinal_features)
uni_comparison_2 = UnivariateSimilarity(output_dir = "TestUniGC/BR_Ordinal_nogroup", logger_name = "BR_Ordinal_nogroup")
scores_2 = uni_comparison_2.get_univariate_score(
original_data = masked_or_data_2.iloc[:,0:47], 
synthetic_data=masked_syn_2.iloc[:,0:47], 
metadata=metadata_2)
scores_df_2 = uni_comparison_2.get_detail_df()
scores_df_2.to_csv("TestUniGC/BR_Ordinal_nogroup/BR_Ordinal_nogroup.csv", index = False)

2025-11-26 21:43:58 - DEBUG - Standalone logger initialized successfully.
2025-11-26 21:43:58 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 47/47 [00:00<00:00, 1839.66it/s]

2025-11-26 21:43:58 - INFO - Univariate similarity score: 0.9356800900880943
2025-11-26 21:43:58 - INFO - Details DataFrame saved to TestUniGC/BR_Ordinal_nogroup/Detail_score_BR_Ordinal_nogroup.csv


2025-11-26 21:44:01 - INFO - Histogram figure saved to TestUniGC/BR_Ordinal_nogroup/BR_Ordinal_nogroup.png


In [50]:
## BR dummy no new columns


# masked_or_data = post_masking(or_data_0)
# masked_or_data_2 = masked_or_data.drop(columns=["PD vs Responder"])
syn_3 = pd.read_csv("gaussiancopula42_BRDummy/gaussiancopula_42_BRDummy.csv", index_col = 0)
masked_syn_3 = post_masking(syn_3)

ordinal_features_3 = ["Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext"]
metadata_3 = MetaData.get_metadata(data = masked_or_data_2, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ordinal_features_3)
uni_comparison_3 = UnivariateSimilarity(output_dir = "TestUniGC/BRDummy", logger_name = "BRDummy")
scores_3 = uni_comparison_3.get_univariate_score(
original_data = masked_or_data_2.iloc[:,0:47], 
synthetic_data=masked_syn_3.iloc[:,0:47], 
metadata=metadata_3)
scores_df_3 = uni_comparison_3.get_detail_df()
scores_df_3.to_csv("TestUniGC/BRDummy/BRDummy.csv", index = False)

2025-11-26 21:54:08 - DEBUG - Standalone logger initialized successfully.
2025-11-26 21:54:08 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 47/47 [00:00<00:00, 1820.80it/s]

2025-11-26 21:54:08 - INFO - Univariate similarity score: 0.9298893972624231
2025-11-26 21:54:08 - INFO - Details DataFrame saved to TestUniGC/BRDummy/Detail_score_BRDummy.csv


2025-11-26 21:54:13 - INFO - Histogram figure saved to TestUniGC/BRDummy/BRDummy.png


In [54]:
## New columns Dummy
masked_or_data = post_masking(or_data_0)
masked_or_data = masked_or_data.rename(columns={"PD vs Responder": "PD_vs_Responder"})
syn_4 = pd.read_csv("gaussiancopula42_RespondersDummy/gaussiancopula_42_RespondersDummy.csv", index_col = 0)
syn_4 = move_last_column_to_first(syn_4)
masked_syn_4 = post_masking(syn_4)
masked_syn_4 = masked_syn_4.rename(columns={"PD vs Responder": "PD_vs_Responder"})


metadata_4 = MetaData.get_metadata(data = masked_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ["BR","Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext"]
                                  )
uni_comparison_4 = UnivariateSimilarity(output_dir = "TestUniGC/RespondersDummy", logger_name = "RespondersDummy")
scores = uni_comparison_4.get_univariate_score(
original_data = masked_or_data.iloc[:,0:48], 
synthetic_data=masked_syn_4.iloc[:,0:48], 
metadata=metadata_4)
scores_df_4 = uni_comparison_4.get_detail_df()
scores_df_4.to_csv("TestUniGC/RespondersDummy/RespondersDummy.csv", index = False)

2025-11-26 22:00:41 - DEBUG - Standalone logger initialized successfully.
2025-11-26 22:00:41 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 1759.99it/s]

2025-11-26 22:00:41 - INFO - Univariate similarity score: 0.9297761630269786
2025-11-26 22:00:41 - INFO - Details DataFrame saved to TestUniGC/RespondersDummy/Detail_score_RespondersDummy.csv


2025-11-26 22:00:43 - INFO - Histogram figure saved to TestUniGC/RespondersDummy/RespondersDummy.png


In [59]:
## New columns ordinal
masked_or_data = post_masking(or_data_0)
masked_or_data = masked_or_data.rename(columns={"PD vs Responder": "PD_vs_Responder"})
syn_5 = pd.read_csv("gaussiancopula42_RespondersOrdinal/gaussiancopula_42_RespondersOrdinal.csv", index_col = 0)
syn_5 = move_last_column_to_first(syn_5)
masked_syn_5 = post_masking(syn_5)
masked_syn_5 = masked_syn_5.rename(columns={"PD vs Responder": "PD_vs_Responder"})


metadata_5 = MetaData.get_metadata(data = masked_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ["BR","Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext","PD_vs_Responder"]
                                  )
uni_comparison_5 = UnivariateSimilarity(output_dir = "TestUniGC/RespondersOrdinal", logger_name = "RespondersOrdinal")
scores = uni_comparison_5.get_univariate_score(
original_data = masked_or_data.iloc[:,0:48], 
synthetic_data=masked_syn_5.iloc[:,0:48], 
metadata=metadata_5)
scores_df_5 = uni_comparison_5.get_detail_df()
scores_df_5.to_csv("TestUniGC/RespondersOrdinal/RespondersOrdinal.csv", index = False)

2025-11-26 22:07:26 - DEBUG - Standalone logger initialized successfully.
2025-11-26 22:07:26 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 1927.75it/s]

2025-11-26 22:07:26 - INFO - Univariate similarity score: 0.9299304258619592
2025-11-26 22:07:26 - INFO - Details DataFrame saved to TestUniGC/RespondersOrdinal/Detail_score_RespondersOrdinal.csv


2025-11-26 22:07:28 - INFO - Histogram figure saved to TestUniGC/RespondersOrdinal/RespondersOrdinal.png


In [61]:
## New columns ordinal
masked_or_data = post_masking(or_data_0)
masked_or_data = masked_or_data.rename(columns={"PD vs Responder": "PD_vs_Responder"})
syn_6 = pd.read_csv("gaussiancopula42_BRRespondersDummy/gaussiancopula_42_BRRespondersDummy.csv", index_col = 0)
syn_6 = move_last_column_to_first(syn_6)
masked_syn_6 = post_masking(syn_6)
masked_syn_6 = masked_syn_6.rename(columns={"PD vs Responder": "PD_vs_Responder"})


metadata_6 = MetaData.get_metadata(data = masked_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ["Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext"]
                                  )
uni_comparison_6 = UnivariateSimilarity(output_dir = "TestUniGC/BRRespondersDummy", logger_name = "BRRespondersDummy")
scores = uni_comparison_6.get_univariate_score(
original_data = masked_or_data.iloc[:,0:48], 
synthetic_data=masked_syn_6.iloc[:,0:48], 
metadata=metadata_6)
scores_df_6 = uni_comparison_6.get_detail_df()
scores_df_6.to_csv("TestUniGC/BRRespondersDummy/BRRespondersDummy.csv", index = False)

2025-11-26 22:55:02 - DEBUG - Standalone logger initialized successfully.
2025-11-26 22:55:02 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 48/48 [00:00<00:00, 1415.34it/s]

2025-11-26 22:55:02 - INFO - Univariate similarity score: 0.9271201562040097
2025-11-26 22:55:02 - INFO - Details DataFrame saved to TestUniGC/BRRespondersDummy/Detail_score_BRRespondersDummy.csv


2025-11-26 22:55:06 - INFO - Histogram figure saved to TestUniGC/BRRespondersDummy/BRRespondersDummy.png


In [65]:
## New columns ordinal
masked_or_data = post_masking(or_data_0)
masked_or_data = masked_or_data.drop(columns = ["BR"])
masked_or_data = masked_or_data.rename(columns={"PD vs Responder": "PD_vs_Responder"})
syn_7 = pd.read_csv("gaussiancopula42_noBR/gaussiancopula_42_noBR.csv", index_col = 0)
syn_7 = move_last_column_to_first(syn_7)
masked_syn_7 = post_masking(syn_7)
masked_syn_7 = masked_syn_7.rename(columns={"PD vs Responder": "PD_vs_Responder"})


metadata_7 = MetaData.get_metadata(data = masked_or_data, 
                                 threshold_unique_values = 10, 
                                    ordinal_features = ["Mstage", "Tx_Start_ECOG",
                       "numPriorTherapies","biopsyContext"]
                                  )
uni_comparison_7 = UnivariateSimilarity(output_dir = "TestUniGC/noBR", logger_name = "noBR")
scores = uni_comparison_7.get_univariate_score(
original_data = masked_or_data.iloc[:,0:47], 
synthetic_data=masked_syn_7.iloc[:,0:47], 
metadata=metadata_7)
scores_df_7 = uni_comparison_7.get_detail_df()
scores_df_7.to_csv("TestUniGC/noBR/noBR.csv", index = False)

2025-11-26 23:01:20 - DEBUG - Standalone logger initialized successfully.
2025-11-26 23:01:20 - INFO - Starting univariate similarity computation.


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 47/47 [00:00<00:00, 1889.02it/s]

2025-11-26 23:01:20 - INFO - Univariate similarity score: 0.9269738491278897
2025-11-26 23:01:20 - INFO - Details DataFrame saved to TestUniGC/noBR/Detail_score_noBR.csv


2025-11-26 23:01:24 - INFO - Histogram figure saved to TestUniGC/noBR/noBR.png
